# Installing Net2Brain

<img src="workshops/data/Net2Brain_Logo.png" width="25%" />

In [19]:
#!pip install -U git+https://github.com/cvai-roig-lab/Net2Brain

# Step 1: Feature Extraction

## Using `FeatureExtractor` with a model from Net2Brain

The FeatureExtractor class provides an interface for extracting features from a given model. When initializing this class, you can customize its behavior by setting various parameters:

- `model` (required): The model from which you want to extract features. Either string in combination with a netset (next parameter), or a variable with a model-type.
- `netset` (optional): The netset (collection of networks) that the model belongs to.
- `device` (optional): The device on which to perform the computations, e.g., 'cuda' for GPU or 'cpu' for CPU. Default is None, which will use the device specified in the global PyTorch settings.
- `pretrained` (optional): A boolean flag indicating whether to use a pretrained model (if available) or to initialize the model with random weights. Default is True, which means that a pretrained model will be used if possible.

- - -


First we need to a dataset to play around with. For that we will use the dataset by [Micheal F. Bonner (2017)](https://www.pnas.org/doi/full/10.1073/pnas.1618228114), which we can download using the `load_dataset` function

In [1]:
from net2brain.utils.download_datasets import DatasetBonnerPNAS2017, DatasetAlgonauts_NSD_Shared
from pprint import pprint
import os

#paths_bonner = DatasetBonnerPNAS2017.load_dataset()
#pprint(paths_bonner)

paths_NSD_shared  = DatasetAlgonauts_NSD_Shared.load_dataset()
#pprint(paths_NSD_shared)
#paths_NSD_shared.keys()


stimuli_path = paths_NSD_shared["images"]

# count files
# num_images = len([f for f in os.listdir(stimuli_path) if f.endswith(('.jpg', '.png'))])

#print("Number of images:", num_images)

### Initating FeatureExtractor


To extract the activations of a pretrained model from a netset, you can use the FeatureExtractor class. First, you need to initialize the class by providing the name of the model and the name of the netset. You can find a suitable model and netset by exploring the taxonomy options available in the Net2Brain toolbox, as shown in the previous notebook "0_Exploring_Net2Brain". For instance, in the following example, we will use AlexNet from the standard netset.

In [2]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='ViT-L/14', netset='Clip', device='cpu')

KeyboardInterrupt: 

The `extract` method computes feature extraction from an image dataset. It takes in the following parameters:

- `data_path` (required): The path to the images from which to extract the features. The images must be in JPEG or PNG format.
- `save_path` (optional): The path to save the extracted features to. If None, the folder where the features are saved is named after the current date in the format "{year}{month}{day}{hour}{minute}".
- `layers_to_extract` (optional): Either provide the list of layers to extract from the model, or a single string 
option from "all", "top_level", and "json", to extract all layers, only the top level blocks or predefined layers 
from a json file. Defaults to "top_level".
- `consolidate_per_layer` (optional): The features are extracted image-wise. This is defaulted to true and will 
consolidate them per layer if not set to False. Defaults to True.
- `dim_reduction` (optional): Type of dimensionality reduction to apply to the extracted features. Choose from `srp` (Sparse Random Projection) and `pca` (Principal Component Analysis). Defaults to None.
- `n_samples_estim`: The number of samples used for estimating the dimensionality reduction. Defaults to 100.
- `n_components` (optonal): Number of components for dimensionality reduction. If None, the number of components is estimated. Defaults to 10,000 (good value for SRP, not PCA).
- `max_dim_allowed` (optional): The threshold over which the dimensionality reduction is applied. If None, it is always applied. Defaults to None.

In [7]:

fx.extract(data_path=stimuli_path, save_path='CLIP_Feat_nsd', consolidate_per_layer=False)

C:\Users\louai\net2brain-env\lib\site-packages\net2brain\architectures\clip_models.py:29: UserWarning: This is a multimodal model. With unimodal input the text-transformer layers will have the same activations throughout all images.
  warnings.warn("This is a multimodal model. With unimodal input the text-transformer layers will have the same activations throughout all images.")
Processing files: 100%|████████████████████████████████████████████████| 872/872 [32:55<00:00,  2.27s/it, subfiles=1/1]


In [1]:
# Extract CLS features into a new folder
import numpy as np
import os
import shutil

input_path = "CLIP_Feat_nsd"
output_path = "CLIP_Feat_nsd_cls"

# overwrite safely
if os.path.exists(output_path):
    shutil.rmtree(output_path)
os.makedirs(output_path)

for file in os.listdir(input_path):
    data = np.load(os.path.join(input_path, file))
    
    arr = data["visual.ln_post"]   # (1, 1024)
    vec = arr[0]                   # (1024,)
    
    np.savez(os.path.join(output_path, file), cls=vec)


this create one RDM because there's one CLS layer.  cannot analyze: model hierarchy vs brain hierarchy ONLY final embedding vs brain  
ALexNet → multiple layers → multiple RDMs  
JEPA: many checkpoints / layers → many RDMs


testing to know why RSA giving errors

In [5]:
data = np.load(os.path.join(input_path, os.listdir(input_path)[0]))

arr = data["visual.ln_post"]

print("Shape:", arr.shape)

Shape: (1, 1024)


In [11]:
import numpy as np
import os

input_path = "CLIP_Feat_nsd"

vars = []

for file in os.listdir(input_path)[:50]:  # sample
    data = np.load(os.path.join(input_path, file))
    
    arr = data["visual.ln_post"]   # (1, ?, 1024)
    
    print("Shape:", arr.shape)
    
    vec = arr.flatten()
    vars.append(np.var(vec))

print("Mean variance:", np.mean(vars))

Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Shape: (1, 1024)
Mean variance: 1.0473477


In [2]:
#exploring feats
import numpy as np
import os

feat_path = "CLIP_Feat_nsd"

layer_file = os.listdir(feat_path)[0]
data = np.load(os.path.join(feat_path, layer_file))

# see what's inside
print("Keys:", data.files)

# load first array
arr = data[data.files[0]]

print("Shape:", arr.shape)
print("Variance:", np.var(arr))

Keys: ['visual.conv1', 'visual.ln_pre', 'visual.transformer.resblocks.0', 'visual.transformer.resblocks.1', 'visual.transformer.resblocks.2', 'visual.transformer.resblocks.3', 'visual.transformer.resblocks.4', 'visual.transformer.resblocks.5', 'visual.transformer.resblocks.6', 'visual.transformer.resblocks.7', 'visual.transformer.resblocks.8', 'visual.transformer.resblocks.9', 'visual.transformer.resblocks.10', 'visual.transformer.resblocks.11', 'visual.transformer.resblocks.12', 'visual.transformer.resblocks.13', 'visual.transformer.resblocks.14', 'visual.transformer.resblocks.15', 'visual.transformer.resblocks.16', 'visual.transformer.resblocks.17', 'visual.transformer.resblocks.18', 'visual.transformer.resblocks.19', 'visual.transformer.resblocks.20', 'visual.transformer.resblocks.21', 'visual.transformer.resblocks.22', 'visual.transformer.resblocks.23', 'visual.ln_post', 'token_embedding', 'transformer.resblocks.0', 'transformer.resblocks.1', 'transformer.resblocks.2', 'transformer

In [7]:
import numpy as np
import os

file = os.listdir("CLIP_Feat_nsd")[0]
data = np.load(os.path.join("CLIP_Feat_nsd", file))

for key in list(data.files)[:2]:
    print(key, data[key].shape)

visual.conv1 (1, 1024, 16, 16)
visual.ln_pre (1, 257, 1024)


In [8]:
arr = data["visual.ln_pre"]
cls = arr[0, 0]

print(cls.shape)
print(cls[:10])

(1024,)
[ 0.03467137 -0.04001092 -0.16942087 -0.03924888  0.00642417  0.02214196
  0.07546216 -0.03680423  0.02232345  0.05064739]


alexnet

In [5]:
import numpy as np
import os

feat_path = "AlexNet_Feat_nsd"

file = os.listdir(feat_path)[0]
data = np.load(os.path.join(feat_path, file))

print("File:", file)
print("Keys (layers):", data.files)

# inspect one layer
key = data.files[0]
arr = data[key]

print("\nLayer name:", key)
print("Shape:", arr.shape)
print("Min:", arr.min(), "Max:", arr.max())
print("Mean:", arr.mean(), "Var:", arr.var())

File: train-0350_nsd-02950.npz
Keys (layers): ['features.0', 'features.3', 'features.6', 'classifier.1']

Layer name: features.0
Shape: (1, 64, 55, 55)
Min: 0.0 Max: 22.056728
Mean: 0.83395696 Var: 2.862747


__Net2Brain__ chooses by default to extract from the top level blocks of the model. You can inspect 
which layers are selected by default by calling the `layers_to_extract` attribute:

In [26]:
fx.layers_to_extract

['layer1', 'layer2', 'layer3', 'layer4']

These are not all the layers that **can** be extracted. If you want to see all the layers that can possibly be extracted you you call `get_all_layers()`.

In [35]:
fx.get_all_layers()

['',
 'features',
 'features.0',
 'features.1',
 'features.2',
 'features.3',
 'features.4',
 'features.5',
 'features.6',
 'features.7',
 'features.8',
 'features.9',
 'features.10',
 'features.11',
 'features.12',
 'avgpool',
 'classifier',
 'classifier.0',
 'classifier.1',
 'classifier.2',
 'classifier.3',
 'classifier.4',
 'classifier.5',
 'classifier.6']

If you wish to change the layers to be extracted you can add it to the `extract` function like with the parameter 
```
fx.extract(..., layers_to_extract=[your_layers])
```
or layers_to_extract="all" to extract from all layers.

- - - 

- - -

# Adding dimensionality reduction
If you wish you can also reduce the dimensionality of the extracted feautures using:

- `srp` (Sparse Random Projection)
- `pca` (Principal Component Analysis)
   - Can apply SRP before PCA for very high-dimensional data with `srp_before_pca=True`

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='AlexNet', netset='Standard', device='cpu')
fx.extract(data_path=stimuli_path, save_path='AlexNet_Feat_dim_red', dim_reduction="srp", n_components=50)

If you want to save the original features to disk, but still want to reduce dimensionality for your analyses, this is 
also possible further down the pipeline when the features are loaded (see 2_RDM_Creation notebook). In that case, set
 `dim_reduction` to None in the extract function.

---

---

# Extracting Features from MMAction Video Models

When extracting features from the video models available in the MMAction netset, there are some 
additional options to keep in mind.
 
Also, as noted in the first notebook, you need to separately 
install MMAction using the provided script in the Net2Brain Github repository, `install_mmaction.sh`.

First, let's now switch to using a video neural dataset, such as the BoldMomentsDataset from 
[Lahner et al.](https://www.nature.com/articles/s41467-024-50310-3):

In [ ]:
from net2brain.utils.download_datasets import DatasetBoldMoments
paths_bmd = DatasetBoldMoments.load_dataset()
stimuli_path = paths_bmd["stimuli_train"]
roi_path = paths_bmd["roi_path"]

And load a video model from the MMAction netset:

In [ ]:
from net2brain.feature_extraction import FeatureExtractor
fx = FeatureExtractor(model='videomae_v2_B', netset='MMAction', device='cpu')

Now, when extracting features you can decide whether and how to aggregate model features across 
time, using the keyword argument `agg_frames`. The options are 'all', 'within_clips', 
'across_clips' or None, and the default is 'across_clips'. 

Here is what each option means:
- None: No aggregation is performed - this is useful to obtain the original per-timestep features
 for subsequent temporally resolved analyses. Shape (1, N_Clips, T, C, H, W).
- 'all': All model feature timesteps are averaged together, resulting in a single time-averaged 
feature vector per video. Shape (1, 1, 1, C, H, W).
- 'within_clips': Model feature timesteps are averaged within each video subclip (model 
processing horizon), resulting in one feature vector per subclip. Shape (1, N_Clips, 1, C, H, W).
- 'across_clips': Model feature timesteps are averaged across subclips, resulting in a single 
time-resolved feature vector per video. Shape (1, 1, T, C, H, W).

In [ ]:
fx.extract(data_path=stimuli_path, save_path='VideoModel_TimeResolvedFeat', agg_frames=None)  # Useful if you want to do temporally resolved analyses later on

---

---

# Extracting Features from Large Language Models

We have also added optionality to extract features from Large Language Models (LLMs) using .txt files. For this you just enter the path to your .txt files, in which each new line represents one new sentence.

Since the feautures are saved per file, and since a .txt file might contain multiple sentences, you can `consolidate_per_txt_file()` in order to seperate each sentence into its own .npz file!

In [ ]:
from net2brain.feature_extraction import FeatureExtractor


extractor = FeatureExtractor("facebook/bart-base", "Huggingface", device="cpu")
layers_to_extract = extractor.get_all_layers()
print(layers_to_extract)

extractor.extract(data_path="textinput_folder", 
                  save_path="LLM_output",
                  consolidate_per_layer=True)


- - -

- - -

# Extracting Features from Audio Models


You can also extract features from audio models using any kind of audio data e.g. *.wav* or *.mp3* files. For this you just enter the path to your audio files. Currently, we provide a selection of CNN based models and Transformer based models. You can find the available models in the taxonomy notebook. For each model group there are mutliple models available, that differ in many design choices like the time window used or the number of layers. Please refer to the paper of the model for more details:
* CNN-based models: [PANNS](https://arxiv.org/pdf/1912.10211)
* Transformer-based models: [AST](https://arxiv.org/pdf/2104.01778)

In the future we will also add more models to this list.

In [ ]:
from net2brain.feature_extraction import FeatureExtractor


extractor = FeatureExtractor(model='PANNS_Cnn10', netset='Audio', device='cpu')
print(extractor.layers_to_extract)

extractor.extract(data_path="../net2brain/tests/audios",
                  save_path="Audio_output",
                  consolidate_per_layer=True)

- - - 

- - -

## Using `FeatureExtractor` with your own DNN

You can also incorporate your own custom model with __Net2Brain__. To do this, supply the `FeatureExtractor` with the following components:

1. Your model
2. An existing netset to fall back to (e.g. Standard, Clip, MMAction) when loading the data and applying standard 
functions.
3. Optionally, your custom transform function (if not provided, standard ImageNet transformations will be used)
4. Optionally your custom extraction function (if not provided, standard Torchextractor will be used)
5. Optionally, your custom feature cleaner (if not provided, no cleaning will be done)
6. The specific layers you want to extract features from

In [18]:
from torchvision import models
from net2brain.feature_extraction import FeatureExtractor

# Define a model
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)  # This one exists in the toolbox as well, it is just supposed to be an example!


# Define extractor 
fx = FeatureExtractor(model=model, device='cpu')

# Run extractor
fx.extract(data_path=stimuli_path,  save_path='ResNet50_Feat', layers_to_extract=['layer1', 'layer2', 'layer3', 'layer4'])

NameError: name 'FeatureExtractor' is not defined

Here an example with your custom functions. Make sure the parameters of your custom function match the ones here.

In [3]:
from torchvision import transforms as T
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor

def my_preprocessor(image, model_name, device):
    """
    Args:
        image (Union[Image.Image, List[Image.Image]]): A PIL Image or a list of PIL Images.
        model_name (str): The name of the model, used to determine specific preprocessing if necessary.
        device (str): The device to which the tensor should be transferred ('cuda' for GPU, 'cpu' for CPU).

    Returns:
        Union[torch.Tensor, List[torch.Tensor]]: The preprocessed image(s) as PyTorch tensor(s).
    """

    transforms = trn.Compose([
        trn.Resize((224, 224)),
        trn.ToTensor(),
        trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = transforms(image).unsqueeze(0)
    if device == 'cuda':
        img_tensor = img_tensor.cuda()

    return img_tensor


def my_extactor(preprocessed_data, layers_to_extract, model):
    # Create a extractor instance
    extractor_model = tx.Extractor(model, layers_to_extract)
    
    # Extract actual features
    _, features = extractor_model(preprocessed_data)

    return features

#def my_extactor_jepa(preprocessed_data, layers_to_extract, model):
    #features = model(preprocessed_data)
    #return {"jepa": features}
    
def my_transformer_extractor(preprocessed_data, layers_to_extract, model):
    # call the ORIGINAL ViT model (not the wrapper) and returns features of every layer
    
    outputs = model.model(preprocessed_data, output_hidden_states=True)  # outputs all layers too. You completely bypass JEPAFeatureExtractor.forward()
    # if it was model() then it uses JEPAFeatureExtractor.
    hidden_states = outputs.hidden_states

    features = {}
    for i, layer in enumerate(hidden_states):
        layer = layer[:, 1:, :]      # remove CLS
        layer = layer.mean(dim=1)    # pool
        features[f"layer_{i}"] = layer

    return features
    
def my_laste_layer_transformer_extractor(preprocessed_data, layers_to_extract, model):
    # call the ORIGINAL ViT model (not the wrapper) 
    outputs = model.model(preprocessed_data)  

    tokens = outputs.last_hidden_state
    tokens = tokens[:, 1:, :]
    features = tokens.mean(dim=1)

    return features
def my_cleaner(features):
    return features



In [ ]:

# Define a model
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)  # This one exists in the toolbox as well, it is just supposed to be an example!

## Define extractor (Note: NO NETSET NEEDED HERE)
fx = FeatureExtractor(model=model, device='cpu', preprocessor=my_preprocessor, feature_cleaner=my_cleaner, extraction_function=my_extactor)

# Run extractor
fx.extract(stimuli_path, layers_to_extract=['layer1', 'layer2', 'layer3', 'layer4'])

# JEPA

load JEPA model

In [7]:
from safetensors.torch import load_file

state_dict = load_file("JEPA_model_ijepa_vith14_1k.safetensors")

print(len(state_dict))
print(list(state_dict.keys()))


517
['embeddings.patch_embeddings.projection.bias', 'embeddings.patch_embeddings.projection.weight', 'embeddings.position_embeddings', 'encoder.layer.0.attention.attention.key.bias', 'encoder.layer.0.attention.attention.key.weight', 'encoder.layer.0.attention.attention.query.bias', 'encoder.layer.0.attention.attention.query.weight', 'encoder.layer.0.attention.attention.value.bias', 'encoder.layer.0.attention.attention.value.weight', 'encoder.layer.0.attention.output.dense.bias', 'encoder.layer.0.attention.output.dense.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.layernorm_after.bias', 'encoder.layer.0.layernorm_after.weight', 'encoder.layer.0.layernorm_before.bias', 'encoder.layer.0.layernorm_before.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.attention.attention.key.bias', 'encoder.layer.1.attention.attention.key.weight', 'encoder.layer.1.attention.attention.qu

JEPA doesn’t rely on a CLS token “summary token” (global representation of the image)
It learns representations differently (predictive masking)

In [8]:
#add extra fake token in jepa model architecture so it matches ViT

import torch

# get JEPA position embeddings
pos_embed = state_dict["embeddings.position_embeddings"]  # (1, 256, 1280)

# create fake CLS token (zeros)
cls_token = torch.zeros(1, 1, pos_embed.shape[-1])

# concatenate → (1, 257, 1280)
new_pos_embed = torch.cat([cls_token, pos_embed], dim=1)

# replace in state_dict
state_dict["embeddings.position_embeddings"] = new_pos_embed

In [9]:
# loading jepa weights into vit
from transformers import ViTModel
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# create empty ViT (same structure)
vit_model = ViTModel.from_pretrained("google/vit-huge-patch14-224-in21k")

# load JEPA weights
missing, unexpected = vit_model.load_state_dict(state_dict, strict=False)
# what its doing: for key in state_dict: if key matches model parameter: copy tensor into model
# if not in jepa (missing) : keep from vit. if not in vit (unexpected), leave
#this is only encoder part. JEPA (from paper) has extra: context encoder, target encoder, predictor

print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

vit_model = vit_model.to(device)
vit_model.eval()



Missing keys: 3
Unexpected keys: 0


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-31): 32 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=1280, out_features=1280, bias=True)
            (key): Linear(in_features=1280, out_features=1280, bias=True)
            (value): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=1280, out_features=1280, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=1280, out_features=5120, bias=True)
          (intermediate_act_fn): GELUA

In [5]:
#for i, (name, module) in enumerate(vit_model.named_modules()):
#    print(i, name, module)
print([n for n, _ in vit_model.named_children()])

['embeddings', 'encoder', 'layernorm', 'pooler']


In [7]:
for i, (name, module) in enumerate(vit_model.named_modules()):
    print(i, name)
#output layer is 583 layernorm



0 
1 embeddings
2 embeddings.patch_embeddings
3 embeddings.patch_embeddings.projection
4 embeddings.dropout
5 encoder
6 encoder.layer
7 encoder.layer.0
8 encoder.layer.0.attention
9 encoder.layer.0.attention.attention
10 encoder.layer.0.attention.attention.query
11 encoder.layer.0.attention.attention.key
12 encoder.layer.0.attention.attention.value
13 encoder.layer.0.attention.attention.dropout
14 encoder.layer.0.attention.output
15 encoder.layer.0.attention.output.dense
16 encoder.layer.0.attention.output.dropout
17 encoder.layer.0.intermediate
18 encoder.layer.0.intermediate.dense
19 encoder.layer.0.intermediate.intermediate_act_fn
20 encoder.layer.0.output
21 encoder.layer.0.output.dense
22 encoder.layer.0.output.dropout
23 encoder.layer.0.layernorm_before
24 encoder.layer.0.layernorm_after
25 encoder.layer.1
26 encoder.layer.1.attention
27 encoder.layer.1.attention.attention
28 encoder.layer.1.attention.attention.query
29 encoder.layer.1.attention.attention.key
30 encoder.layer.1.a

attention
→ residual
→ layernorm_before
→ MLP/intermediate
→ output.dense
→ dropout
→ residual
→ layernorm_after


x→embeddings→32 transformer layers→final norm→pooling  
each encoder layer: Attention → MLP → residual connections 
MLP (Multilayer perceptron): nonlinear transformation of each token
state_dict:	raw tensors (numbers)  
named_modules():	functions / operations that use those tensors (f(x))  
Each layer defines a different embedding space: f_l(x) So thesis is basically: Which f_l best matches brain representations?

In [9]:
print(missing[:10])
print(unexpected[:10])

['embeddings.cls_token', 'pooler.dense.weight', 'pooler.dense.bias']
[]


try forward pass to extract feature

In [11]:
import torch

dummy = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    outputs = vit_model(dummy)

tokens = outputs.last_hidden_state  # (1, 257, 1280)

# remove CLS token (first token)
tokens = tokens[:, 1:, :]  # (1, 256, 1280)

# average pooling → final feature vector
features = tokens.mean(dim=1)  # (1, 1280)

print(features.shape)

torch.Size([1, 1280])


In [13]:
# see whic layer is the output of outputs.last_hidden_state
# if true its  583 layernorm
activations = {}

def hook_fn(name):
    def hook(module, input, output):
        activations[name] = output
    return hook

vit_model.layernorm.register_forward_hook(
    hook_fn("final_layernorm")
)
# we can also test if its 582 encoder.layer.31.layernorm_after:  vit_model.encoder.layer[31].layernorm_after.register_forward_hook(


outputs = vit_model(dummy)

torch.allclose(
    outputs.last_hidden_state,
    activations["final_layernorm"]
)


True

the model gives (B, 257, 1280) but Net2Brain expects (B, 1280)  
Standardizes JEPA to behave like: model(x) → (B, D)

In [7]:
import torch.nn as nn

class JEPAFeatureExtractor(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model #You completely bypass JEPAFeatureExtractor.forward()

    def forward(self, x):
        with torch.no_grad():
            outputs = self.model(x) #Run the actual transformer. 

        tokens = outputs.last_hidden_state  # (B, 257, 1280). Get all patch representations

        # remove fake CLS token
        tokens = tokens[:, 1:, :]

        # average pooling
        features = tokens.mean(dim=1)  # (B, 1280)

        return features

jepa_extractor = JEPAFeatureExtractor(vit_model)

In [8]:
# varify model(image_batch) → (B, D)
out = jepa_extractor(torch.randn(2,3,224,224).to(device))
print(out.shape)

torch.Size([2, 1280])


model = "what network to run"  
extraction_function = "how to get features out of that network"  
RSA analysis used outputs.hidden_states, NOT outputs.last_hidden_state.
| Model | Used for RSA                          |
| ----- | ------------------------------------- |
| JEPA  | `hidden_states`                       |
| MAE   | `hidden_states`                       |
| DINO  | should use intermediate block outputs |
| iBOT  | should match this too                 |


In [16]:
# add jepa extractor
fx = FeatureExtractor(
    model=jepa_extractor,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_transformer_extractor
)
fx.extract(
    data_path=stimuli_path,
    save_path='JEPA_Feat_nsd_2'
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [56:56<00:00,  3.92s/it, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:16<00:00, 11.43it/s]


test rdms shape

In [5]:

import os
import numpy as np

brain_rdms_path = os.path.join(paths_NSD_shared["root"], "brain_data")

files = os.listdir(brain_rdms_path)
print("Number of files:", len(files))
print("First 5 files:", files[:5])

# check shapes of a few RDMs
for f in files[:5]:
    data = np.load(os.path.join(brain_rdms_path, f))
    
    # get the array inside the .npz
    key = list(data.keys())[0]
    arr = data[key]
    
    print(f"{f} -> shape: {arr.shape}")

Number of files: 35
First 5 files: ['EBA_lh_RDMs.npz', 'EBA_rh_RDMs.npz', 'FBA-2_rh_RDMs.npz', 'FFA-1_lh_RDMs.npz', 'FFA-1_rh_RDMs.npz']
EBA_lh_RDMs.npz -> shape: (8, 872, 872)
EBA_rh_RDMs.npz -> shape: (8, 872, 872)
FBA-2_rh_RDMs.npz -> shape: (8, 872, 872)
FFA-1_lh_RDMs.npz -> shape: (8, 872, 872)
FFA-1_rh_RDMs.npz -> shape: (8, 872, 872)


# LeJepa

In [2]:
from safetensors.torch import load_file
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint_path = r"C:\Users\louai\Net2Brain\notebooks\LeJepa training\lejepa_epoch_100.pth"

from dataclasses import dataclass

@dataclass
class LeJEPAConfig:
    pass
    
ckpt = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

state_dict = ckpt["model_state_dict"]


filtered_state_dict = {}

for k, v in state_dict.items():

    # keep ONLY backbone weights
    if k.startswith("backbone."):

        new_key = k.replace("backbone.", "")
        filtered_state_dict[new_key] = v

state_dict = filtered_state_dict

In [13]:

print(ckpt.keys())
print(list(ckpt["model_state_dict"].keys())[:30])

dict_keys(['epoch', 'model_state_dict', 'optimizer_state_dict', 'history', 'config'])
['backbone.cls_token', 'backbone.pos_embed', 'backbone.patch_embed.proj.weight', 'backbone.patch_embed.proj.bias', 'backbone.blocks.0.norm1.weight', 'backbone.blocks.0.norm1.bias', 'backbone.blocks.0.attn.qkv.weight', 'backbone.blocks.0.attn.qkv.bias', 'backbone.blocks.0.attn.proj.weight', 'backbone.blocks.0.attn.proj.bias', 'backbone.blocks.0.norm2.weight', 'backbone.blocks.0.norm2.bias', 'backbone.blocks.0.mlp.fc1.weight', 'backbone.blocks.0.mlp.fc1.bias', 'backbone.blocks.0.mlp.fc2.weight', 'backbone.blocks.0.mlp.fc2.bias', 'backbone.blocks.1.norm1.weight', 'backbone.blocks.1.norm1.bias', 'backbone.blocks.1.attn.qkv.weight', 'backbone.blocks.1.attn.qkv.bias', 'backbone.blocks.1.attn.proj.weight', 'backbone.blocks.1.attn.proj.bias', 'backbone.blocks.1.norm2.weight', 'backbone.blocks.1.norm2.bias', 'backbone.blocks.1.mlp.fc1.weight', 'backbone.blocks.1.mlp.fc1.bias', 'backbone.blocks.1.mlp.fc2.weight

In [17]:
import timm

timm.list_models("*vit*small*")

['convit_small',
 'vit_small_patch16_224',
 'vit_small_patch16_224_in21k',
 'vit_small_patch16_384',
 'vit_small_patch32_224',
 'vit_small_patch32_224_in21k',
 'vit_small_patch32_384',
 'vit_small_r26_s32_224',
 'vit_small_r26_s32_224_in21k',
 'vit_small_r26_s32_384',
 'vit_small_resnet26d_224',
 'vit_small_resnet50d_s16_224']

In [3]:
# get backbone only
filtered_state_dict = {}

for k, v in ckpt["model_state_dict"].items():

    if k.startswith("backbone."):

        new_key = k.replace("backbone.", "")
        filtered_state_dict[new_key] = v

# create timm ViT
import timm

vit_model = timm.create_model(
    "vit_small_patch16_224",
    pretrained=False
)

#because: you do NOT want classifier head, only backbone embeddings for RSA
filtered_state_dict.pop("pos_embed", None)
filtered_state_dict.pop("head.weight", None)
filtered_state_dict.pop("head.bias", None)


# load LeJEPA backbone weights
missing, unexpected = vit_model.load_state_dict(
    filtered_state_dict,
    strict=False
)

print("Missing:", len(missing))
print("Unexpected:", len(unexpected))

vit_model = vit_model.to(device)
vit_model.eval()

C:\Users\louai\net2brain-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Missing: 3
Unexpected: 0


VisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    (norm): Identity()
  )
  (pos_drop): Dropout(p=0.0, inplace=False)
  (blocks): Sequential(
    (0): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
    )
    (1): Block(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine

In [8]:
from torchvision import transforms as T
from torchvision import transforms as trn
import torchextractor as tx
from net2brain.feature_extraction import FeatureExtractor

def my_extactor_lejepa(preprocessed_data, layers_to_extract, model):

    features = {}

    x = model.model.patch_embed(preprocessed_data)

    cls_token = model.model.cls_token.expand(x.shape[0], -1, -1)
    x = torch.cat((cls_token, x), dim=1)

    x = x + model.model.pos_embed
    x = model.model.pos_drop(x)

    for i, block in enumerate(model.model.blocks):

        x = block(x)

        tokens = x[:, 1:, :]          # remove CLS
        pooled = tokens.mean(dim=1)  # average pooling

        features[f"layer_{i}"] = pooled

    return features

In [9]:
jepa_extractor = JEPAFeatureExtractor(vit_model)

In [10]:
# add jepa extractor
fx = FeatureExtractor(
    model=jepa_extractor,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    preprocessor=my_preprocessor,
    feature_cleaner=my_cleaner,
    extraction_function=my_extactor_lejepa
)
fx.extract(
    data_path=stimuli_path,
    save_path='LEJEPA_Feat_nsd'
)

Processing files: 100%|████████████████████████████████████████████████| 872/872 [02:42<00:00,  5.37it/s, subfiles=1/1]


Consolidating data per layer...


100%|████████████████████████████████████████████████████████████████████████████████| 872/872 [01:01<00:00, 14.25it/s]
